In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import os

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src import setup_workspace, query_to_df, ds3on1

# 1. Inicializar workspace y rutas
paths = setup_workspace()

# 2. Consultar directamente a PostgreSQL (vía DuckDB ATTACH)
#df = query_to_df("SELECT * FROM db.EuInfo;")

# 3. Análisis univariable 3-en-1
#ds3on1(df, 'happy', Desc=True, Stat=True, Graph=True)

✅ Workspace anclado en: /home/josu/Documentos/DataScienceCourse
📂 Ruta de tablas verificada: /home/josu/Documentos/DataScienceCourse/data/raw/Tables


🟢 < Module 10.1 > Model Serialization, REST API Backend (Flask) and Standalone HTML<>JS Client
### 🎯 Context & Objectives

A Machine Learning pipeline living inside a Jupyter Notebook delivers zero operational value until external systems or end-users can query it reliably. In traditional prototypes (such as monolithic Streamlit scripts), model inference is tightly coupled to the UI and often suffers from positional array mismatches (`ValueError: feature_names mismatch`) when different target models require distinct subsets of selected features[.

In this exercise, you will decouple your validated predictive model from Module 9 into a **client-server production architecture**
1. Serialize your trained `ImbPipeline` artifact using portable relative paths (`pathlib`).
2. Build a lightweight **Flask REST API** (`src/app.py`) that validates incoming JSON payloads **by exact feature name** against `model.feature_names_in_`.
3. Pair it with a standalone **HTML5/Vanilla JavaScript client** (`web/index.html`) featuring numeric inputs/sliders, asynchronous `fetch()` calls, and a graceful client-side simulation fallback mode.

---

### 📑 Exercise Instructions

#### 1. Syntactic / Algorithmic Level (Artifact Serialization & Name-Aligned Flask API)

- **Portable Pipeline Serialization (`joblib` & `pathlib`):**
    - From your Module 9 notebook, export your best trained `ImbPipeline` (e.g., Hybrid Logistic Regression or Gradient Boosting) to `models/best_ess_pipeline.joblib` using `joblib.dump()`.
    - Resolve all filesystem paths dynamically via `pathlib.Path(__file__).resolve().parents[1]`—zero hardcoded Windows or Linux user paths.
    - Ensure `flask` and `flask-cors` are available in your active `.venv-core` environment (extending `requirements/ReqJupML.txt`).

- **RESTful Inference Contract (`src/app.py`):**
    - Enable CORS (`flask_cors.CORS(app)`) so a locally opened `web/index.html` file in Chromium can communicate with `http://127.0.0.1:5000` without browser origin blocks.
    - Expose `GET /health` returning the loaded model status, estimator class name, and the exact ordered list of expected input features (`model.feature_names_in_`).
    - Expose `POST /predict` accepting a JSON object of feature key-value pairs:
        - **Strict Feature-Name Validation:** Check that all keys in `model.feature_names_in_` exist in the incoming JSON and contain valid numeric values. If any key is missing or invalid, return HTTP `400 Bad Request` with a descriptive JSON error listing `missing_features` or `invalid_types`
        - **DataFrame Construction:** Build a 1-row `pd.DataFrame` ordered strictly by `list(model.feature_names_in_)` (never by raw positional lists) before calling `.predict()` and `.predict_proba()`.
        - Return HTTP `200 OK` with a structured JSON payload (`prediction`, `probability`, `label`, `features_used`).

---

#### 2. Applied / Data Level (Standalone HTML/JS Web Client & API Testing)

- **Interactive Parameter Controls (`web/index.html`):**
    - Build a clean, responsive HTML5 interface with sliders (`<input type="range">`) and numeric inputs (`<input type="number">`) matching your trained model's feature contract (e.g., `lrscale`, `institutional_trust_index`, `gdpcapitapct`, `inflationpct`, `unemploymentpct`, `macro_cluster`).
    - Display live numeric readouts next to every slider and provide a "Reset Defaults" button.

- **Asynchronous `fetch()` & Graceful Fallback Mode:**
    - Send an asynchronous `POST` request (`Content-Type: application/json`) to `http://127.0.0.1:5000/predict` when the user submits or adjusts parameters.
    - Wrap the network call in `try / catch`: when the Flask server responds, display the real model probability with a **`LIVE BACKEND`** badge; if unreachable (e.g., static preview on GitHub Pages), execute a transparent client-side heuristic and display a **`SIMULATION MODE`** warning badge.

- **End-to-End Integration Verification:**
    - Verify the API from terminal (`curl`) or a test script (`requests`) demonstrating:
        1. A valid JSON payload returning HTTP `200` and probability scores.
        2. A malformed JSON payload (omitting one required feature) returning HTTP `400` with the exact missing column name.

---

### 🌐 Official References & Documentation

- **Flask Quickstart & Request Parsing:** [Flask Official Documentation](https://flask.palletsprojects.com/en/latest/quickstart/)
- **Scikit-Learn Model Persistence:** [Model Persistence with Joblib](https://scikit-learn.org/stable/model_persistence.html)
- **Python `pathlib` Standard Library:** [pathlib Documentation](https://docs.python.org/3/library/pathlib.html)
- **MDN Fetch API Guide:** [Using the Fetch API (JavaScript)](https://developer.mozilla.org/en-US/docs/Web/API/Fetch_API/Using_Fetch)

---

### 📚 Recommended Manuals & CheatSheets

- **Serving ML Models in Production:** *Designing Machine Learning Systems (Chip Huyen)* — Chapter 7 (Model Deployment and Prediction Service).
- **Web APIs & JSON Contracts:** *Flask Web Development (Miguel Grinberg)* — Chapter 14 (Application Programming Interfaces).

💻 **Your Challenge:** Serialize your best Module 9 pipeline, implement the name-validated Flask server (`src/app.py`) and standalone client (`web/index.html`), test both `200` and `400` HTTP responses, and share your solution here for code review! 🚀🌐

In [2]:
#1: lets prepare our best model-estimator, with few more variables to make a decent html viewer, for that lets prepare a dataframe, Wich gona have some of their variables transformed and
# preprocessed in order to have an efficient applyment.
# In this demo ill change the model to a continuous one, the model will estimate a range value inside the total range value of our target variable.
# Our variable target will be Happy, but in this case will be a flat number with one decimal from 0 to 10.
# We will apply a backend capable to estimate it for all countries of Europe, so we will use kmeans to macroeconomics variables, wich i dont have all of them.
#Data selection: From our main data from surveys we will chose: Happy (0-10), lrscale(0-10),rlgdgr(0-10, how religious are you), agea(x, integer), 
# gndr(1-2, male=1),hhmmb(x, int, how many ppl lives in your home),cntry("string 2 digits", EU country acronimous, control var) 
# mnactic(1-9,Main Activity: (1)Paid work,Education,UnemplpoyedJobSeeker,UnemployedNonJobSeeker,Permanently sick or disabled,Retired,Community or military service,Housework, looking after children, (9)Other),
# from macro panel : unemploypct(X float, percent of unemployment per country), year(x, year of the records), geo('string 2 digits', state acronimous), macro_region("string", location of the country)
# gdpcapitapct(X, float number of capita rent.), inflationpct(X. resulting percentage inflation over the 10-year time range)
# n_resp(x, integer, audit var,number of non respondent rows,)

import duckdb as db
import numpy as np
import pandas as pd
import random
df101 = query_to_df("""
    WITH joined_data AS (
        SELECT 
            a.cntry,
            a.lrscale,
            a.rlgdgr,
            a.agea,
            a.gndr,
            a.hhmmb,
            a.mnactic,
            a.happy,
            b.gdpcapitapct,
            b.inflationpct,
            b.unemploymentpct,
            b.macro_region,
            ROW_NUMBER() OVER (
                PARTITION BY a.cntry, a.gndr
                ORDER BY hash(a.cntry, a.lrscale, 42)
            ) AS rn
        FROM db.surveys a
        LEFT JOIN db.macro_panel b 
            ON a.cntry = b.geo
           AND b.year = 2024
    )
    SELECT * EXCLUDE (rn)
    FROM joined_data
    WHERE rn <= 250;
""")

print(f"✅ Stratified Dataset: {df101.shape}")
print(df101.groupby(["cntry", "gndr"]).size().unstack(fill_value=0))   

✅ Stratified Dataset: (12000, 12)
gndr     1    2
cntry          
AT     250  250
BE     250  250
BG     250  250
CH     250  250
DE     250  250
EE     250  250
ES     250  250
FI     250  250
FR     250  250
GB     250  250
GR     250  250
HR     250  250
HU     250  250
IE     250  250
IL     250  250
IT     250  250
NL     250  250
NO     250  250
PL     250  250
PT     250  250
RS     250  250
SE     250  250
SI     250  250
SK     250  250


## Target: `happy` (ordinal 0-10) treated as continuous

The variable `happy` is ordinal (0-10 integer scale). For this module, we treat it
as a **quasi-continuous** target to demonstrate regression pipeline mechanics
(scoping, scaling, GroupKFold CV, model serialization, REST inference).

This is a deliberate simplification:
- **Why:** to decouple the course from discrete/ordinal target assumptions and
  exercise the full continuous-prediction workflow (MAE, RMSE, R², float outputs).
- **Limitation:** the true generative process is ordinal. A proper
  ordinal regression (PROBIT/CLOGIT) or quantile regression would be more
  statistically honest. We accept the approximation for pedagogical scope.
- **Practical consequence:** predictions may return values like 6.37, which is
  mathematically valid for the regression model but semantically "between 6 and 7"
  on the original Likert scale. The API client displays this as a continuous
  readout, not a class label.   

In [3]:
df101.info()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   cntry            12000 non-null  str    
 1   lrscale          12000 non-null  int64  
 2   rlgdgr           12000 non-null  int64  
 3   agea             12000 non-null  int64  
 4   gndr             12000 non-null  int64  
 5   hhmmb            12000 non-null  int64  
 6   mnactic          12000 non-null  int64  
 7   happy            12000 non-null  int64  
 8   gdpcapitapct     7500 non-null   float64
 9   inflationpct     7500 non-null   float64
 10  unemploymentpct  7500 non-null   float64
 11  macro_region     7500 non-null   str    
dtypes: float64(3), int64(7), str(2)
memory usage: 1.2 MB


In [4]:
# In this case/exercise we will use 2 criteria for control grouping the dataset, we will avoid to use macro economical data, instead we will use
# "cntry" and control group micro, and   macro_region as macro control variable.

REGION_MAP = {
    "AT": "western", "BE": "western", "CH": "western", "DE": "western",
    "EE": "northern", "ES": "southern", "FI": "northern", "FR": "western",
    "GB": "western", "GR": "southern", "HR": "southern", "HU": "eastern",
    "IE": "western", "IL": "eastern", "IT": "southern", "NL": "western",
    "NO": "northern", "PL": "eastern", "PT": "southern", "RS": "southern",
    "SE": "northern", "SI": "southern", "SK": "eastern", "BG": "southern",
}
CNTRYFC_MAP ={ #EAST COUNTRIES: FROM 0 TO 9, WEST COUNTRIES:FROM 10 TO 19, NORTHERN: FROM 20 TO 29, SOUTHERN: FROM 30 TO 39.
    "AT": "10", "BE": "11", "CH": "12", "DE": "13",
    "EE": "20", "ES": "30", "FI": "21", "FR": "14",
    "GB": "15", "GR": "31", "HR": "32", "HU": "0",
    "IE": "16", "IL": "1", "IT": "33", "NL": "17",
    "NO": "22", "PL": "2", "PT": "34", "RS": "35",
    "SE": "23", "SI": "36", "SK": "3", "BG": "37",
}
REGIONFC_MAP = {#The region number is related to the ten of the country number(East=0,West=1,North=2,South=3)
    "eastern": "0",
    "western": "1",
    "northern": "2",
    "southern": "3",
}

df101["macro_region"] = df101["cntry"].map(REGION_MAP) 
df101["cntryfc"] = df101["cntry"].map(CNTRYFC_MAP) 
df101["regionfc"] = df101["macro_region"].map(REGIONFC_MAP)
df101.info()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   cntry            12000 non-null  str    
 1   lrscale          12000 non-null  int64  
 2   rlgdgr           12000 non-null  int64  
 3   agea             12000 non-null  int64  
 4   gndr             12000 non-null  int64  
 5   hhmmb            12000 non-null  int64  
 6   mnactic          12000 non-null  int64  
 7   happy            12000 non-null  int64  
 8   gdpcapitapct     7500 non-null   float64
 9   inflationpct     7500 non-null   float64
 10  unemploymentpct  7500 non-null   float64
 11  macro_region     12000 non-null  str    
 12  cntryfc          12000 non-null  str    
 13  regionfc         12000 non-null  str    
dtypes: float64(3), int64(7), str(4)
memory usage: 1.4 MB


In [5]:
df1011 = df101[['lrscale','rlgdgr','agea','gndr','hhmmb','mnactic','happy','regionfc','cntryfc']]
del df101
df1011[['regionfc', 'cntryfc']] = df1011[['regionfc', 'cntryfc']].astype(int)   
df1011.info()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   lrscale   12000 non-null  int64
 1   rlgdgr    12000 non-null  int64
 2   agea      12000 non-null  int64
 3   gndr      12000 non-null  int64
 4   hhmmb     12000 non-null  int64
 5   mnactic   12000 non-null  int64
 6   happy     12000 non-null  int64
 7   regionfc  12000 non-null  int64
 8   cntryfc   12000 non-null  int64
dtypes: int64(9)
memory usage: 843.9 KB


In [13]:
# ============================================================
# BLOCK 0: Core imports
# ============================================================

import numpy as np              # Numerical arrays, RNG (default_rng), linear algebra
import pandas as pd             # DataFrame/Series manipulation, dtypes, .loc indexing
import joblib                   # Serialize/deserialize fitted sklearn pipelines to .joblib
from pathlib import Path        # Cross-platform filesystem paths (no hardcoded /home/user/...)
from dataclasses import dataclass, field  # Lightweight spec objects (frozen=True → immutable)
from enum import Enum           # Type-safe categorical labels (VarType, Transform)
from sklearn.model_selection import GroupShuffleSplit  # Create is_test flag via group-aware split
from sklearn.pipeline import Pipeline # Allow us to make a complex function concatenation without touching the original df
                                      # Saving memory and make the feature engineering cleaner.
from scipy.spatial import cKDTree    # Spatial clustering wich allow us to record intervariable classes in percent.
# ========================================================================================================================
# LAYER 1: Preprocess step (split, vartype, non respondent filter, hot-deck impute) → PipelineOutput
# ========================================================================================================================
class VarType(Enum):
    DISCRETE = "discrete"
    CONTINUOUS = "continuous"
    CATEGORICAL = "categorical"
    FLAG = "flag"

@dataclass(frozen=True)
class ColumnSpec:
    name: str
    vtype: VarType

@dataclass(frozen=True)
class DatasetSpec:
    columns: list[ColumnSpec]
    target: str
    group: str
    n_splits: int = 4
    test_size: float = 0.2

    @property
    def feature_cols(self) -> list[str]:
        excluded = {self.target, self.group}
        return [
            c.name for c in self.columns
            if c.vtype in (VarType.DISCRETE, VarType.CONTINUOUS, VarType.CATEGORICAL)
            and c.name not in excluded
        ]

    @property
    def cat_feature_indices(self) -> list[int]:
        cat_names = {c.name for c in self.columns if c.vtype == VarType.CATEGORICAL}
        return [i for i, name in enumerate(self.feature_cols) if name in cat_names]

# ============================================================
# BLOCK 5: PipelineOutput (ampliado con artefactos runtime)
# ============================================================
@dataclass
class PipelineOutput:
    # Data
    X_train: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_test: pd.Series
    groups_train: pd.Series
    groups_test: pd.Series
    spec: DatasetSpec
    # Baseline stats
    y_test_mean: float = 0.0
    y_test_var: float = 0.0
    baseline_mae: float = 0.0
    # Runtime artifacts (for Flask)
    impute_table: dict = field(default_factory=dict)       # hot-deck lookup
    target_dist: dict = field(default_factory=dict)        # percentiles jerárquicos
    knn_data: dict = field(default_factory=dict)           # cKDTree + happy array
    encodings: dict = field(default_factory=dict)          # mapeos frontend↔backend


# ============================================================
# LAYER 1: Wrapper (preprocess + split + build runtime artifacts)
# ============================================================
def prepare_regressor(df: pd.DataFrame, spec: DatasetSpec, seed: int = 42) -> PipelineOutput:

    # --- Internal: create is_test flag ---
    def _pre_split(d):
        d = d.copy()
        if "is_test" not in d.columns:
            gss = GroupShuffleSplit(n_splits=1, test_size=spec.test_size, random_state=seed)
            _, test_idx = next(gss.split(d, d[spec.target], groups=d[spec.group]))
            d["is_test"] = 0
            d.loc[test_idx, "is_test"] = 1
        return d

    # --- Internal: clean & count ---
    def _clean_and_count(d):
        d = d.copy()
        d["n_resp"] = 0
        CODE_GROUPS = [
            [99999, 88888, 77777, 66666],
            [9999,  8888,  7777,  6666],
            [999,   888,   777,   666],
            [99,    88,    77,    66],
        ]
        for col in d.columns:
            if col == "n_resp" or not pd.api.types.is_numeric_dtype(d[col]):
                continue
            unique = set(d[col].dropna().unique())
            codes_found = next(
                ([c for c in g if c in unique] for g in CODE_GROUPS if any(c in unique for c in g)),
                None
            )
            if codes_found:
                mask = d[col].isin(codes_found)
                d.loc[mask, "n_resp"] += 1
                d.loc[mask, col] = np.nan
        return d

    # --- Internal: hot-deck imputation ---
    def _hot_deck(X_tr, X_te, rng):
        X_tr, X_te = X_tr.copy(), X_te.copy()
        protected = {"n_resp"}
        for col in X_tr.columns:
            if col in protected or not pd.api.types.is_numeric_dtype(X_tr[col]):
                continue
            valid_tr = X_tr[col].dropna()
            if valid_tr.empty:
                continue
            weights = valid_tr[col].value_counts(normalize=True) if False else valid_tr.value_counts(normalize=True)
            cats, probs = weights.index.to_numpy(), weights.to_numpy()
            mask_tr, mask_te = X_tr[col].isna(), X_te[col].isna()
            if mask_tr.any():
                X_tr.loc[mask_tr, col] = rng.choice(cats, size=int(mask_tr.sum()), p=probs)
            if mask_te.any():
                X_te.loc[mask_te, col] = rng.choice(cats, size=int(mask_te.sum()), p=probs)
        return X_tr, X_te

    # === MAIN FLOW ===
    rng = np.random.default_rng(seed)

    # 1. Pre-split
    df_split = _pre_split(df)

    # 2. Clean & count
    df_clean = _clean_and_count(df_split)

    # 2.5. Drop target NaN
    n_before = len(df_clean)
    df_clean = df_clean[df_clean[spec.target].notna()].reset_index(drop=True)
    n_dropped = n_before - len(df_clean)
    if n_dropped > 0:
        print(f"⚠️  {n_dropped} rows dropped (target '{spec.target}' is NaN)")

    # 3. Mask
    mask_train = (df_clean["is_test"] == 0)
    mask_test  = (df_clean["is_test"] == 1)

    feat = spec.feature_cols
    X_train = df_clean.loc[mask_train, feat].copy().reset_index(drop=True)
    y_train = df_clean.loc[mask_train, spec.target].astype(float).reset_index(drop=True)
    X_test  = df_clean.loc[mask_test,  feat].copy().reset_index(drop=True)
    y_test  = df_clean.loc[mask_test,  spec.target].astype(float).reset_index(drop=True)

    # 4. Hot-deck
    X_train, X_test = _hot_deck(X_train, X_test, rng)

    # 5. Cast types (derived from spec)
    float_cols = [c.name for c in spec.columns if c.vtype == VarType.CONTINUOUS]
    int_cols   = [c.name for c in spec.columns if c.vtype in (VarType.DISCRETE, VarType.CATEGORICAL)]
    for c in float_cols:
        if c in X_train.columns:
            X_train[c] = X_train[c].astype(float)
            X_test[c]  = X_test[c].astype(float)
    for c in int_cols:
        if c in X_train.columns:
            X_train[c] = X_train[c].astype(int)
            X_test[c]  = X_test[c].astype(int)

    # 6. Groups
    groups_train = df_clean.loc[mask_train, spec.group].astype(int).reset_index(drop=True)
    groups_test  = df_clean.loc[mask_test,  spec.group].astype(int).reset_index(drop=True)

    # 7. Baseline stats
    y_mean = float(y_test.mean())
    y_var  = float(y_test.std() ** 2)
    baseline_mae = float(np.mean(np.abs(y_test.values - y_mean)))
    print(f"\n  y_test mean: {y_mean:.4f} | std: {np.sqrt(y_var):.4f} | var: {y_var:.4f}")
    print(f"  Baseline MAE: {baseline_mae:.4f}")

        # 8. Build impute table (hot-deck lookup for Flask)
    train_full = df_clean.loc[mask_train, feat + [spec.group]].reset_index(drop=True)
    impute_table = {}
    for c, group in train_full.groupby(spec.group):
        for col in feat:
            if not pd.api.types.is_numeric_dtype(train_full[col]):
                continue
            vc = group[col].dropna().value_counts(normalize=True)
            impute_table[(int(c), col)] = (vc.index.to_numpy(), vc.to_numpy())
    # Fallback global
    for col in feat:
        if not pd.api.types.is_numeric_dtype(train_full[col]):
            continue
        vc = train_full[col].dropna().value_counts(normalize=True)
        impute_table[(None, col)] = (vc.index.to_numpy(), vc.to_numpy())   

    # 9. Build target distribution (percentiles jerárquicos)
    target_dist = {"global": y_train.values.copy()}
    target_dist["country"] = {}
    target_dist["region"] = {}
    for c, group in df_clean.groupby(spec.group):
        target_dist["country"][int(c)] = group[spec.target].dropna().values
    if "regionfc" in df_clean.columns:
        for r, group in df_clean.groupby("regionfc"):
            target_dist["region"][int(r)] = group[spec.target].dropna().values

    # 10. Build KNN index (multivariable percentile)
    cat_names = {c.name for c in spec.columns if c.vtype == VarType.CATEGORICAL}
    numeric_feat = [c for c in feat if c not in cat_names]
    X_num = X_train[numeric_feat].values.astype(float)
    X_mean = X_num.mean(axis=0)
    X_std  = X_num.std(axis=0)
    X_std[X_std == 0] = 1  # avoid division by zero
    X_norm = (X_num - X_mean) / X_std
    tree = cKDTree(X_norm)
    knn_data = {
        "tree": tree,
        "X_mean": X_mean,
        "X_std": X_std,
        "happy": y_train.values,
        "numeric_cols": numeric_feat,
    }

        # 11. Encodings (frontend ↔ backend translation)
    
    encodings = {
        "cntry_to_fc": {k: int(v) for k, v in CNTRYFC_MAP.items()},
        "cntry_to_region": {k: int(REGIONFC_MAP[REGION_MAP[k]]) for k in REGION_MAP},
    }

    print(f"\n  Impute table:  {len(impute_table)} keys")
    print(f"  Target dist:   global={len(target_dist['global'])}, "
          f"country={len(target_dist['country'])}, region={len(target_dist['region'])}")
    print(f"  KNN index:     {len(knn_data['happy'])} points, {len(knn_data['numeric_cols'])} numeric dims")
    print(f"  Encodings:     {len(encodings['cntry_to_fc'])} countries mapped")   

    return PipelineOutput(
        X_train, X_test, y_train, y_test,
        groups_train, groups_test, spec,
        y_test_mean=y_mean,
        y_test_var=y_var,
        baseline_mae=baseline_mae,
        impute_table=impute_table,
        target_dist=target_dist,
        knn_data=knn_data,
        encodings=encodings,
    )
    

In [14]:
# ============================================================
# LAYER 2: Configurable Pipeline + GroupKFold + GridSearch
# ============================================================
from sklearn.ensemble import HistGradientBoostingRegressor  # Gradient boosting with native categorical support, no scaling needed
from sklearn.linear_model import Ridge                      # L2-regularized linear regression (baseline)
from sklearn.model_selection import (
    GroupKFold,        # CV that keeps all samples of a group (country) in one fold
    GridSearchCV,      # Exhaustive hyperparameter search using the CV splits above
    cross_validate,    # Return per-fold metrics (MAE, R²) for variance estimation
)
from sklearn.metrics import (
    mean_absolute_error,      # MAE: average absolute prediction error (interpretable in target units)
    mean_squared_error,       # MSE: squared error (penalizes large mistakes more)
    r2_score,                 # R²: proportion of variance explained (1.0 = perfect, 0.0 = mean baseline)
)


MODELS = {
    "gb":    lambda seed: HistGradientBoostingRegressor(random_state=seed),
    "ridge": lambda seed: Ridge(),
}

GRIDS = {
    "gb": {
        "model__learning_rate": [0.05, 0.1, 0.2],
        "model__max_depth": [3, 6, 10],
        "model__max_iter": [100, 300],
    },
    "ridge": {
        "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
    },
}


def run_regressor(output: PipelineOutput, model_type: str = "gb",
                  grid_search: bool = False, seed: int = 42):
    spec = output.spec
    cat_idx = spec.cat_feature_indices

    model = MODELS[model_type](seed)
    if model_type == "gb":
        model.set_params(categorical_features=cat_idx)

    pipe = Pipeline([("model", model)])

    gkf = GroupKFold(n_splits=spec.n_splits)
    cv_splits = list(gkf.split(output.X_train, output.y_train, output.groups_train))

    metrics = {}

    if grid_search and model_type in GRIDS:
        search = GridSearchCV(pipe, GRIDS[model_type], cv=cv_splits,
                              scoring="neg_mean_absolute_error", n_jobs=1, error_score="raise")
        search.fit(output.X_train, output.y_train)
        pipe = search.best_estimator_
        metrics["best_params"] = search.best_params_
        metrics["cv_best_mae"] = round(-search.best_score_, 4)

    cv_res = cross_validate(pipe, output.X_train, output.y_train, cv=cv_splits,
                            scoring=["neg_mean_absolute_error", "r2"])
    metrics["cv_mae"] = f"{-cv_res['test_neg_mean_absolute_error'].mean():.4f} ± {cv_res['test_neg_mean_absolute_error'].std():.4f}"
    metrics["cv_r2"]  = f"{cv_res['test_r2'].mean():.4f} ± {cv_res['test_r2'].std():.4f}"

    pipe.fit(output.X_train, output.y_train)
    y_pred = pipe.predict(output.X_test)

    metrics["test_mae"]  = round(mean_absolute_error(output.y_test, y_pred), 4)
    metrics["test_rmse"] = round(np.sqrt(mean_squared_error(output.y_test, y_pred)), 4)
    metrics["test_r2"]   = round(r2_score(output.y_test, y_pred), 4)
    metrics["baseline_mae"] = round(output.baseline_mae, 4)
    metrics["gain_vs_baseline"] = round(output.baseline_mae - metrics["test_mae"], 4)

    return pipe, metrics


# ============================================================
# PERSISTENCE
# ============================================================
def persist_all(pipe, output: PipelineOutput, name: str, root: Path):
    """Persist model + all runtime artifacts to <root>/models/."""
    models_dir = root / "models"
    models_dir.mkdir(parents=True, exist_ok=True)

    # 1. Model
    model_step = pipe.named_steps["model"]
    model_step.feature_names_in_ = np.array(output.spec.feature_cols)
    joblib.dump(pipe, models_dir / f"{name}.joblib")

    # 2. Impute table
    joblib.dump(output.impute_table, models_dir / f"{name}_impute.joblib")

    # 3. Target distribution
    joblib.dump(output.target_dist, models_dir / f"{name}_target_dist.joblib")

    # 4. KNN index
    joblib.dump(output.knn_data, models_dir / f"{name}_knn.joblib")

    # 5. Encodings
    joblib.dump(output.encodings, models_dir / f"{name}_encodings.joblib")

    print(f"\n✅ Persisted to {models_dir}/:")
    for f in sorted(models_dir.glob(f"{name}*")):
        print(f"   {f.name:40s} {f.stat().st_size / 1024:.1f} KB")



In [15]:
# ============================================================
# EXECUTION
# ============================================================
spec_reg = DatasetSpec(
    columns=[
        ColumnSpec("lrscale",    VarType.CONTINUOUS),
        ColumnSpec("rlgdgr",     VarType.CONTINUOUS),
        ColumnSpec("agea",       VarType.DISCRETE),
        ColumnSpec("gndr",       VarType.CATEGORICAL),
        ColumnSpec("hhmmb",      VarType.DISCRETE),
        ColumnSpec("mnactic",    VarType.CATEGORICAL),
        ColumnSpec("regionfc",   VarType.CATEGORICAL),
        ColumnSpec("cntryfc",    VarType.CATEGORICAL),
        ColumnSpec("n_resp",     VarType.DISCRETE),
    ],
    target="happy",
    group="cntryfc",
    n_splits=4,
    test_size=0.2,
)

# Layer 1
prep = prepare_regressor(df1011, spec_reg)
print(f"\nTrain: {prep.X_train.shape} | Test: {prep.X_test.shape}")
print(f"Target range: [{prep.y_train.min():.0f}, {prep.y_train.max():.0f}]")

# Layer 2
pipe, metrics = run_regressor(prep, model_type="gb", grid_search=True, seed=42)

print(f"\n{'='*50}")
for k, v in metrics.items():
    print(f"  {k}: {v}")
print(f"{'='*50}")

# Persist all
persist_all(pipe, prep, "best_happy_regressor", paths["ROOT"])

⚠️  44 rows dropped (target 'happy' is NaN)

  y_test mean: 7.5599 | std: 1.7776 | var: 3.1599
  Baseline MAE: 1.3653

  Impute table:  160 keys
  Target dist:   global=9461, country=24, region=4
  KNN index:     9461 points, 5 numeric dims
  Encodings:     24 countries mapped

Train: (9461, 8) | Test: (2495, 8)
Target range: [0, 10]

  best_params: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 100}
  cv_best_mae: 1.3306
  cv_mae: 1.3306 ± 0.0636
  cv_r2: 0.0777 ± 0.0250
  test_mae: 1.3163
  test_rmse: 1.7195
  test_r2: 0.0639
  baseline_mae: 1.3653
  gain_vs_baseline: 0.049

✅ Persisted to /home/josu/Documentos/DataScienceCourse/models/:
   best_happy_regressor.joblib              123.9 KB
   best_happy_regressor_encodings.joblib    0.3 KB
   best_happy_regressor_impute.joblib       52.6 KB
   best_happy_regressor_knn.joblib          642.3 KB
   best_happy_regressor_target_dist.joblib  262.6 KB
